# Phase 2a: NER Preprocessing

Converts `ner_bio_dataset.json` (tokens + BIO tags) into padded numeric
arrays ready for the BiLSTM-CRF model.

**Includes character-level features alongside word-level ones.** This
matters for generalization: your NER model will eventually see prescription
text with drug names it never saw during training (e.g. Indian brand names
later on). Word-level embeddings alone map unknown words to a single `UNK`
token and lose all information. Character-level features let the model
learn *sub-word patterns* (capitalization, common suffixes like "-olol",
"-pril", "-azole" typical of drug names) so it can make a reasonable guess
even on completely unseen words — instead of being hardcoded to only
recognize the exact drug names it was trained on.

**Output:**
- `X_words.npy`, `X_chars.npy`, `y_tags.npy` — padded train arrays
- `X_words_val.npy`, `X_chars_val.npy`, `y_tags_val.npy` — padded val arrays
- `word_vocab.json`, `char_vocab.json`, `tag_vocab.json` — vocabularies
- `preprocessing_config.json` — MAX_LEN, MAX_WORD_LEN etc., needed again
  when preprocessing new inference input later


## 1. Setup

In [1]:
%pip install numpy scikit-learn -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

INPUT_JSON = "../data/annotated/ner_bio_dataset.json"
OUTPUT_DIR = "../data/annotated/preprocessed"

with open(INPUT_JSON, encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records)} sentences.")

Loaded 689 sentences.


## 2. Train / validation split

Split at the sentence level before building vocab, so the vocab is built
only from training data — this is important: if we built vocab from all
data including validation, we'd be leaking validation-only words into the
model's "known vocabulary," making validation performance look artificially
better than real-world performance on genuinely new text.

In [3]:
train_records, val_records = train_test_split(
    records, test_size=0.15, random_state=42
)

print(f"Train: {len(train_records)} sentences")
print(f"Val:   {len(val_records)} sentences")

Train: 585 sentences
Val:   104 sentences


## 3. Build vocabularies (from TRAINING data only)

- **Word vocab**: lowercased words, with `<PAD>`=0 and `<UNK>`=1 reserved
- **Char vocab**: individual characters seen in training tokens, same
  reserved indices
- **Tag vocab**: all BIO tags seen, plus a `<PAD>` tag for padded positions
  (important: the loss function needs to ignore/mask padded positions, and
  having a distinct PAD tag makes that masking straightforward)

In [4]:
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"

word_counter = Counter()
char_counter = Counter()
tag_set = set()

for rec in train_records:
    for tok in rec["tokens"]:
        word_counter[tok.lower()] += 1
        char_counter.update(tok)
    tag_set.update(rec["tags"])

word_vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word, _ in word_counter.most_common():
    word_vocab[word] = len(word_vocab)

char_vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for ch, _ in char_counter.most_common():
    char_vocab[ch] = len(char_vocab)

tag_vocab = {"<PAD>": 0}
for tag in sorted(tag_set):
    tag_vocab[tag] = len(tag_vocab)

print(f"Word vocab size: {len(word_vocab)}")
print(f"Char vocab size: {len(char_vocab)}")
print(f"Tag vocab size:  {len(tag_vocab)}")
print(f"Tags: {list(tag_vocab.keys())}")

Word vocab size: 1417
Char vocab size: 78
Tag vocab size:  12
Tags: ['<PAD>', 'B-DIAGNOSIS', 'B-DOSAGE', 'B-DRUG', 'B-DURATION', 'B-FREQUENCY', 'I-DIAGNOSIS', 'I-DOSAGE', 'I-DRUG', 'I-DURATION', 'I-FREQUENCY', 'O']


## 4. Choose padding lengths

`MAX_LEN` (tokens per sentence) and `MAX_WORD_LEN` (characters per token)
are picked from the 95th percentile of training data, not the max — a few
extreme outlier sentences shouldn't force every other sentence to be padded
with excessive filler, which wastes compute and can dilute the signal.

In [5]:
sentence_lengths = [len(rec["tokens"]) for rec in train_records]
word_lengths = [len(tok) for rec in train_records for tok in rec["tokens"]]

MAX_LEN = int(np.percentile(sentence_lengths, 95))
MAX_WORD_LEN = int(np.percentile(word_lengths, 95))

print(f"Sentence length: min={min(sentence_lengths)}, max={max(sentence_lengths)}, "
      f"95th percentile (MAX_LEN)={MAX_LEN}")
print(f"Word length: min={min(word_lengths)}, max={max(word_lengths)}, "
      f"95th percentile (MAX_WORD_LEN)={MAX_WORD_LEN}")

Sentence length: min=5, max=104, 95th percentile (MAX_LEN)=46
Word length: min=1, max=21, 95th percentile (MAX_WORD_LEN)=9


## 5. Encoding functions (word, char, tag → indices, with padding/truncation)

In [6]:
def encode_words(tokens, vocab, max_len):
    ids = [vocab.get(tok.lower(), vocab[UNK_TOKEN]) for tok in tokens]
    ids = ids[:max_len]
    ids += [vocab[PAD_TOKEN]] * (max_len - len(ids))
    return ids

def encode_chars(tokens, vocab, max_len, max_word_len):
    char_ids = []
    for tok in tokens[:max_len]:
        c_ids = [vocab.get(ch, vocab[UNK_TOKEN]) for ch in tok][:max_word_len]
        c_ids += [vocab[PAD_TOKEN]] * (max_word_len - len(c_ids))
        char_ids.append(c_ids)
    # pad sentence-level: add empty (all-PAD) token rows if sentence < max_len
    while len(char_ids) < max_len:
        char_ids.append([vocab[PAD_TOKEN]] * max_word_len)
    return char_ids

def encode_tags(tags, vocab, max_len):
    ids = [vocab.get(t, vocab["<PAD>"]) for t in tags][:max_len]
    ids += [vocab["<PAD>"]] * (max_len - len(ids))
    return ids

## 6. Build the full padded arrays for train + val

In [7]:
def build_arrays(records, word_vocab, char_vocab, tag_vocab, max_len, max_word_len):
    X_words, X_chars, y_tags = [], [], []
    for rec in records:
        X_words.append(encode_words(rec["tokens"], word_vocab, max_len))
        X_chars.append(encode_chars(rec["tokens"], char_vocab, max_len, max_word_len))
        y_tags.append(encode_tags(rec["tags"], tag_vocab, max_len))
    return np.array(X_words), np.array(X_chars), np.array(y_tags)

X_words_train, X_chars_train, y_tags_train = build_arrays(
    train_records, word_vocab, char_vocab, tag_vocab, MAX_LEN, MAX_WORD_LEN
)
X_words_val, X_chars_val, y_tags_val = build_arrays(
    val_records, word_vocab, char_vocab, tag_vocab, MAX_LEN, MAX_WORD_LEN
)

print("Train shapes:", X_words_train.shape, X_chars_train.shape, y_tags_train.shape)
print("Val shapes:  ", X_words_val.shape, X_chars_val.shape, y_tags_val.shape)

Train shapes: (585, 46) (585, 46, 9) (585, 46)
Val shapes:   (104, 46) (104, 46, 9) (104, 46)


## 7. Sanity check: decode one example back to verify correctness

In [8]:
inv_word_vocab = {v: k for k, v in word_vocab.items()}
inv_tag_vocab = {v: k for k, v in tag_vocab.items()}

sample_idx = 0
print("Original tokens:", train_records[sample_idx]["tokens"][:12])
print("Original tags:  ", train_records[sample_idx]["tags"][:12])
print()
decoded_words = [inv_word_vocab[i] for i in X_words_train[sample_idx][:12]]
decoded_tags = [inv_tag_vocab[i] for i in y_tags_train[sample_idx][:12]]
print("Decoded words (lowercased + padded):", decoded_words)
print("Decoded tags:                       ", decoded_tags)

Original tokens: ['RX', ':', 'Micronase', '2.5', 'mg', 'Tab', 'PO', 'QAM', '#', '30', ',', 'Bactrim']
Original tags:   ['O', 'O', 'B-DRUG', 'B-DOSAGE', 'I-DOSAGE', 'O', 'O', 'O', 'O', 'O', 'O', 'B-DRUG']

Decoded words (lowercased + padded): ['rx', ':', 'micronase', '2.5', 'mg', 'tab', 'po', 'qam', '#', '30', ',', 'bactrim']
Decoded tags:                        ['O', 'O', 'B-DRUG', 'B-DOSAGE', 'I-DOSAGE', 'O', 'O', 'O', 'O', 'O', 'O', 'B-DRUG']


## 8. Save everything for the model-training notebook

In [9]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.save(f"{OUTPUT_DIR}/X_words_train.npy", X_words_train)
np.save(f"{OUTPUT_DIR}/X_chars_train.npy", X_chars_train)
np.save(f"{OUTPUT_DIR}/y_tags_train.npy", y_tags_train)
np.save(f"{OUTPUT_DIR}/X_words_val.npy", X_words_val)
np.save(f"{OUTPUT_DIR}/X_chars_val.npy", X_chars_val)
np.save(f"{OUTPUT_DIR}/y_tags_val.npy", y_tags_val)

with open(f"{OUTPUT_DIR}/word_vocab.json", "w") as f:
    json.dump(word_vocab, f)
with open(f"{OUTPUT_DIR}/char_vocab.json", "w") as f:
    json.dump(char_vocab, f)
with open(f"{OUTPUT_DIR}/tag_vocab.json", "w") as f:
    json.dump(tag_vocab, f)

config = {
    "MAX_LEN": MAX_LEN,
    "MAX_WORD_LEN": MAX_WORD_LEN,
    "word_vocab_size": len(word_vocab),
    "char_vocab_size": len(char_vocab),
    "tag_vocab_size": len(tag_vocab),
    "n_train": len(train_records),
    "n_val": len(val_records),
}
with open(f"{OUTPUT_DIR}/preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved all arrays + vocabs + config -> {OUTPUT_DIR}/")
print(config)

Saved all arrays + vocabs + config -> ../data/annotated/preprocessed/
{'MAX_LEN': 46, 'MAX_WORD_LEN': 9, 'word_vocab_size': 1417, 'char_vocab_size': 78, 'tag_vocab_size': 12, 'n_train': 585, 'n_val': 104}


## What's next

Move to the next notebook: **Phase 2b — BiLSTM-CRF model build + training**.
It will load these `.npy` arrays and vocab files directly, build a model
with:
- A word-embedding branch (learned from `word_vocab`)
- A character-level BiLSTM branch (learned from `char_vocab`) — this is
  what gives the model a fighting chance on drug names it's never seen
- Both branches concatenated → BiLSTM → CRF output layer

Then train, and evaluate F1 per entity class on the validation set.
